# HSRE Model Statistics

Inspect configured checkpoints using local cache only. Parameter counts are architecture statistics, not accuracy or benchmark results. The models are not weight-loaded in this notebook.

In [ ]:
import gc
import importlib.metadata as metadata
import json
import os
import sys

import torch
from huggingface_hub import try_to_load_from_cache
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForAudioClassification,
    AutoModelForSpeechSeq2Seq,
)

MODEL_SPECS = [
    ("Whisper Small", "openai/whisper-small", AutoModelForSpeechSeq2Seq),
    ("HuBERT Base", "facebook/hubert-base-ls960", AutoModel),
    (
        "Emotion checkpoint",
        "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition",
        AutoModelForAudioClassification,
    ),
    ("BGE-M3 (configured, not used by intent path)", "BAAI/bge-m3", None),
]
WEIGHT_FILES = (
    "model.safetensors",
    "pytorch_model.bin",
    "model.safetensors.index.json",
    "pytorch_model.bin.index.json",
)


def cached_weight_paths(model_id):
    paths = []
    for filename in WEIGHT_FILES:
        path = try_to_load_from_cache(model_id, filename)
        if isinstance(path, str):
            if filename.endswith(".index.json"):
                with open(path, encoding="utf-8") as index_file:
                    shard_names = sorted(set(json.load(index_file)["weight_map"].values()))
                for shard_name in shard_names:
                    shard_path = try_to_load_from_cache(model_id, shard_name)
                    if isinstance(shard_path, str):
                        paths.append(shard_path)
            else:
                paths.append(path)
    return sorted(set(paths))


def model_row(display_name, model_id, model_class):
    paths = cached_weight_paths(model_id)
    row = {
        "Model": display_name,
        "Checkpoint": model_id,
        "Weights cached": bool(paths),
        "Checkpoint size (MB)": round(
            sum(os.path.getsize(path) for path in paths) / 1_000_000, 2
        ),
        "Architecture": "not inspected",
        "Parameters": "not available",
        "Labels": "not available",
    }
    if model_class is None or not paths:
        return row

    config = AutoConfig.from_pretrained(model_id, local_files_only=True)
    with torch.device("meta"):
        model = model_class.from_config(config)
    row["Architecture"] = type(model).__name__
    row["Parameters"] = sum(parameter.numel() for parameter in model.parameters())
    labels = getattr(config, "id2label", None)
    if labels:
        row["Labels"] = ", ".join(str(labels[key]) for key in sorted(labels))
    del model
    gc.collect()
    return row


rows = [model_row(*spec) for spec in MODEL_SPECS]
print(
    f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | "
    f"Transformers {metadata.version('transformers')}"
)
columns = (
    "Model",
    "Checkpoint",
    "Weights cached",
    "Checkpoint size (MB)",
    "Architecture",
    "Parameters",
    "Labels",
)
widths = {
    column: max(len(column), *(len(str(row.get(column, ""))) for row in rows))
    for column in columns
}
print(" | ".join(column.ljust(widths[column]) for column in columns))
print("-+-".join("-" * widths[column] for column in columns))
for row in rows:
    print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

print(
    "\nCounts come from model architecture configs instantiated on the meta device. "
    "Weights are not loaded, audio is not processed, and accuracy is not measured."
)